## Testing HDF5 results caching

In [ ]:
import h5py
import numpy as np
import time
from collections import defaultdict
import os
import logging
from src.logging_util import handler

log_level = logging.DEBUG
logging.getLogger('src').setLevel(logging.DEBUG)
logging.basicConfig(level=log_level, handlers=[handler], force=True)
logger = logging.getLogger(__name__)
TEST_DIR = 'test2'
# Generate test data
def generate_batch(batch_size=1000, sequences=10, max_idxs=10000 ):
    sequences_out = np.random.choice(sequences, batch_size).astype(str)  # 10 sequences
    indices = np.random.choice(max_idxs, batch_size, replace=False)
    data = np.random.rand(batch_size, 5, 5, 256).astype(np.float32)
    return sequences_out, indices, data

# Method 1: Naive approach
def store_naive(sequences, indices, data):
    start = time.time()
    for seq, idx, arr in zip(sequences, indices, data):
        with h5py.File(TEST_DIR+'/test_naive.h5', 'a') as f:
            if seq not in f:
                f.create_group(seq)
            f[seq][str(idx)] = arr
    return time.time() - start


# Method 2: Batched with open file
def store_batched(sequences, indices, data):
    start = time.time()
    with h5py.File(TEST_DIR+'/test_batched.h5', 'a') as f:
        for seq, idx, arr in zip(sequences, indices, data):
            seq_g = f.require_group(seq)
            grp = seq_g.require_group(str(idx))
            grp['x_gen'] = arr
    return time.time() - start

# Method 3: Grouped by sequence
def store_grouped(sequences, indices, data):
    start = time.time()
    # Group by sequence
    seq_groups = defaultdict(list)
    for seq, idx, arr in zip(sequences, indices, data):
        seq_groups[seq].append((idx, arr))
    
    with h5py.File(TEST_DIR+'/test_grouped.h5', 'a') as f:
        for seq, items in seq_groups.items():
            seq_g = f.require_group(seq)
            for idx, arr in items:
                grp = seq_g.require_group(str(idx))
                grp['x_gen'] = arr
    return time.time() - start


def store_auto(sequences, indices, data):
    start = time.time()
    # Group by sequence
    with h5py.File(TEST_DIR+'/test_auto.h5', 'a') as f:
        for seq, idx, arr in zip(sequences, indices, data):
            f.create_dataset(f'{seq}/{idx}/x_gen',data=arr, dtype="f")
    return time.time() - start

# Test them
if not os.path.exists(TEST_DIR):
    os.makedirs(TEST_DIR)
else:
    for fname in os.listdir(TEST_DIR):
        file_path = os.path.join(TEST_DIR, fname)
        if os.path.isfile(file_path):
            os.remove(file_path)
logger.info("Generating test data...")
sequences, indices, data = generate_batch(10000, 100, 100000)
logger.info("Test data generated with %d sequences.", len(sequences))
# logger.debug("Soring naive.")
# time_naive = store_naive(sequences, indices, data)
logger.debug("Storing batched.")
time_batched = store_batched(sequences, indices, data)
logger.debug("Storing grouped.")
time_grouped = store_grouped(sequences, indices, data)
logger.debug("Storing auto.")
time_auto = store_auto(sequences, indices, data)
logger.debug("Done")
# print(f"Naive (1000 items): {time_naive:.3f}s")
print(f"Batched (1000 items): {time_batched:.3f}s") 
print(f"Grouped (1000 items): {time_grouped:.3f}s")
print(f"Auto (1000 items): {time_auto:.3f}s")

In [ ]:
import random
import time

def random_lookup_time(h5_path, sequences, indices, n_lookups=1000):
    seqs = sequences.tolist()
    idxs = indices.tolist()
    pairs = list(zip(seqs, idxs))
    random.shuffle(pairs)
    pairs = pairs[:n_lookups]
    start = time.time()
    results = []
    with h5py.File(h5_path, 'r') as f:
        for seq, idx in pairs:
            res = f[str(seq)][str(idx)].get('x_gen', "empty")
            print(type(res))
            results.append(res)
            
    print(np.stack(results))
    return time.time() - start

# def random_append_time(h5_path, sequences, indices, n_lookups=1000):
#     seqs = sequences.tolist()
#     idxs = indices.tolist()
#     pairs = list(zip(seqs, idxs))
#     random.shuffle(pairs)
#     pairs = pairs[:n_lookups]
#     start = time.time()
#     with h5py.File(h5_path, 'r') as f:
#         for seq, idx in pairs:
#             # Read the existing array
#             arr = f[str(seq)][str(idx)][:]

#             # Generate a new random array of shape (1, 7, 256)
#             new_arr = np.random.rand(1, 7, 256).astype(arr.dtype)

#             # Concatenate along the first dimension
#             arr_appended = np.concatenate([arr, new_arr], axis=0)

#             # Overwrite the dataset with the new array
#             del f[str(seq)][str(idx)]
#             f[str(seq)].create_dataset(str(idx), data=arr_appended)
#     return time.time() - start

n_lookups = 3
lookup_times = {}
for fname in ['test_batched.h5', 'test_grouped.h5', 'test_auto.h5']:
    h5_path = f'{TEST_DIR}/{fname}'
    t = random_lookup_time(h5_path, sequences, indices, n_lookups=n_lookups)
    lookup_times[fname] = t
    print(f"Random lookup time for {fname} ({n_lookups} lookups): {t:.3f}s")